# 扩散与粘度分析

| 文件 | 用途 |
|------|------|
| `log.lammps` | 温度 / 压力 / 密度 / 运行粘度 |
| `result_traj_trans.lammpstrj` | 疏轨迹（unwrap）→ 平动 |
| `result_traj_rot.lammpstrj` | 密短轨迹 → 转动 |
| `result_viscosity_correlate.dat` | 应力自相关 → GK 粘度 |

- `_helper_functions.py` — `load_lammps_universe` / thermo / `save_nglview_frame`
- `_translational_diffusion.py` — `mode="selection"` / `"com"`
- `_rotational_diffusion.py` — 偶极轴 / **完整姿态 body**（C1 + 测地线 + unwrap）
- `_viscosity_analysis.py` — GK 积分


In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "font.size": 10,
})
from MDAnalysis import transformations as trans
import nglview as nv
import warnings
warnings.filterwarnings("ignore")

from _helper_functions import (
    load_lammps_universe,
    read_result_thermo,
    move_origin_to_corner,
    save_nglview_frame,
)
import _translational_diffusion as tdiff
import _rotational_diffusion as rdiff
import _viscosity_analysis as visc_an

# ---- 本例参数（换体系改这里）----
TOPO = "./result_atoms.eq.data"
N_MOL = 500
RESNAME = "H2O"
SELECT_O = "type 2"      # 或 "element O"
SELECT_H = "type 1"      # 或 "element H"
DT_TRANS_FS = 500.0
DT_ROT_FS = 20.0
# 调试用前 N 帧；正式报数都改 None
TRANS_N_FRAMES = 1000     # 平动疏轨迹前 N 帧（500×0.5 ps ≈ 250 ps）
ROT_N_FRAMES = 1000       # 转动密轨迹前 N 帧（500×20 fs ≈ 10 ps）


## 1. 温度与压力

读全部 thermo 块；生产段合并最后两块（密轨迹前后两段 `run`）。


In [ ]:
thermo_all = read_result_thermo("log.lammps", segment=None)
print("thermo blocks / rows:", thermo_all["segment"].nunique(), len(thermo_all))

prod = read_result_thermo("log.lammps", segment=-1)
try:
    prev = read_result_thermo("log.lammps", segment=-2)
    if "viscosity_cp" in prev.columns and prev["viscosity_cp"].notna().any():
        prod = pd.concat([prev, prod], ignore_index=True)
except Exception:
    pass

fig, axes = plt.subplots(2, 2, figsize=(8.5, 4.8))
for seg, g in thermo_all.groupby("segment"):
    t_ns = g["time"].to_numpy() / 1e6
    axes[0, 0].plot(t_ns, g["temp"], lw=1.0, label=f"seg{seg}")
    axes[0, 1].plot(t_ns, g["press"], lw=1.0, label=f"seg{seg}")
axes[0, 0].set_ylabel("T [K]")
axes[0, 0].set_xlabel("time in segment [ns]")
axes[0, 0].tick_params(direction="in")
axes[0, 1].set_ylabel("P [atm]")
axes[0, 1].set_xlabel("time in segment [ns]")
axes[0, 1].tick_params(direction="in")

t_prod = prod["time"].to_numpy() / 1e6
axes[1, 0].plot(t_prod, prod["density"], "-")
axes[1, 0].axhline(1.0, color="k", ls="--", lw=0.8)
axes[1, 0].set_ylabel(r"density [g/cm$^3$]")
axes[1, 0].set_xlabel("production time [ns]")
axes[1, 0].tick_params(direction="in")

if "viscosity_cp" in prod.columns:
    axes[1, 1].plot(t_prod, prod["viscosity_cp"], "-")
    axes[1, 1].set_ylabel(r"$\eta$ running [cP]")
else:
    axes[1, 1].text(
        0.5, 0.5, "no viscosity_cP", ha="center", va="center",
        transform=axes[1, 1].transAxes,
    )
axes[1, 1].set_xlabel("production time [ns]")
axes[1, 1].tick_params(direction="in")

fig.tight_layout()
fig.savefig("thermo_TP.png", bbox_inches="tight")
plt.show()
print(
    f"production mean T = {prod['temp'].mean():.2f} K, "
    f"P = {prod['press'].mean():.1f} atm, "
    f"density = {prod['density'].mean():.4f} g/cm^3"
)


## 2. 轨迹可视化

仅显示时 `wrap`；后面扩散分析须保持 unwrap，另开 Universe。


In [ ]:
u_view = load_lammps_universe(
    TOPO, "./result_traj_trans.lammpstrj",
    dt_fs=DT_TRANS_FS, n_residues=N_MOL, resname=RESNAME,
)
u_view.trajectory.add_transformations(
    move_origin_to_corner,
    trans.wrap(u_view.atoms, compound="residues"),
)
view = nv.show_mdanalysis(u_view)
view.clear_representations()
view.add_representation(selection=RESNAME, repr_type="licorice", radius=".4")
view.add_unitcell()
view


In [ ]:
# 导出某一帧为 PNG（默认最后一帧）
# save_nglview_frame(view, "last_frame.png")  # frame=-1 → last


## 3. 平动扩散

同一 unwrap 轨迹上：原子 selection（本例氧）与分子 COM。

`TRANS_N_FRAMES` 非空时只读前 N 帧，便于调试。


In [ ]:
u_trans = load_lammps_universe(
    TOPO, "./result_traj_trans.lammpstrj",
    dt_fs=DT_TRANS_FS, n_residues=N_MOL, resname=RESNAME,
)

trans_O = tdiff.translational_diffusion(
    u_trans, mode="selection", select=SELECT_O,
    dt_fs=DT_TRANS_FS, n_frames=TRANS_N_FRAMES, t_min_ps=20.0,
)
trans_COM = tdiff.translational_diffusion(
    u_trans, mode="com", select="all", compound="residues",
    dt_fs=DT_TRANS_FS, n_frames=TRANS_N_FRAMES, t_min_ps=20.0,
)

fig, ax = plt.subplots(figsize=(4.5, 3.2))
for res, lab, color in (
    (trans_O, "O", "tab:blue"),
    (trans_COM, "COM", "tab:orange"),
):
    t, msd, sl = res["times_ps"], res["msd_A2"], res["fit_slice"]
    ax.plot(t, msd, "-", color=color, label=lab)
    slope = 6.0 * res["D_A2_ps"]
    tf = t[sl]
    ax.plot(
        tf, slope * tf + (msd[sl][0] - slope * tf[0]),
        "--", color=color, lw=1,
        label=rf"$D_\mathrm{{{lab}}}={res['D_A2_ps']:.3f}\,\mathrm{{Å}}^2/\mathrm{{ps}}$",
    )
ax.set_xlabel("$t$ [ps]")
ax.set_ylabel(r"MSD [Å$^2$]")
ax.tick_params(direction="in")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig("msd_translational.png", bbox_inches="tight")
plt.show()
print(
    f"[trans] frames={trans_O['n_frames_used']} "
    f"(TRANS_N_FRAMES={TRANS_N_FRAMES!r})"
)
print(f"O:   {trans_O['D_1e5_cm2_s']:.3f} ×10⁻⁵ cm²/s")
print(f"COM: {trans_COM['D_1e5_cm2_s']:.3f} ×10⁻⁵ cm²/s")


## 4. 转动扩散（偶极轴）

`orientation="sites"`：偶极 = O → mean(H)。三种量：$C_1/C_2$、测地线 $\langle\theta^2\rangle$（$\approx 4 D_r t$）、unwrap $\langle|\Phi|^2\rangle$。

`ROT_N_FRAMES` 非空时只读前 N 帧，便于调试。


In [ ]:
u_rot = load_lammps_universe(
    TOPO, "./result_traj_rot.lammpstrj",
    dt_fs=DT_ROT_FS, n_residues=N_MOL, resname=RESNAME,
)

rot_res = rdiff.rotational_diffusion(
    u_rot,
    orientation="sites",
    origin_select=SELECT_O,
    target_select=SELECT_H,
    n_target=2,
    dt_fs=DT_ROT_FS,
    n_frames=ROT_N_FRAMES,
    c1_t_min_ps=0.5, c1_t_max_ps=5.0,
    amsd_t_min_ps=0.2, amsd_t_max_ps=2.0,
)

t = rot_res["times_ps"]
c1, c2 = rot_res["C1"], rot_res["C2"]
amsd, amsd_u = rot_res["angular_msd"], rot_res["angular_msd_unwrapped"]
sl = rot_res["fit_slice_c1"]
sl2 = rot_res["fit_slice_amsd"]
sl_u = rot_res["fit_slice_amsd_unwrapped"]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 3.2))
ax0.plot(t, c1, "-", label=r"$C_1$")
ax0.plot(t, c2, "-", label=r"$C_2$")
tt = t[sl]
ax0.plot(
    tt, rot_res["C1_amp"] * np.exp(-2.0 * rot_res["Dr_C1_per_ps"] * tt),
    "k--", lw=1,
    label=rf"$D_r(C_1)={rot_res['Dr_C1_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
ax0.set_xlabel("$t$ [ps]")
ax0.set_ylabel("orientation correlation")
ax0.set_xlim(0, min(20, t[-1]))
ax0.tick_params(direction="in")
ax0.legend(frameon=False, fontsize=8)

ax1.plot(t, amsd, "-", label=r"geodesic $\langle\theta^2\rangle$")
ax1.plot(t, amsd_u, "-", alpha=0.75, label=r"unwrap $\langle|\Phi|^2\rangle$")
slope = 4.0 * rot_res["Dr_amsd_per_ps"]
tt2 = t[sl2]
ax1.plot(
    tt2, slope * tt2 + (amsd[sl2][0] - slope * tt2[0]),
    "k--", lw=1,
    label=rf"$D_r(\theta^2)={rot_res['Dr_amsd_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
slope_u = 4.0 * rot_res["Dr_amsd_unwrapped_per_ps"]
ttu = t[sl_u]
ax1.plot(
    ttu, slope_u * ttu + (amsd_u[sl_u][0] - slope_u * ttu[0]),
    ":", color="tab:orange", lw=1.2,
    label=rf"$D_r(\Phi)={rot_res['Dr_amsd_unwrapped_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
ax1.set_xlabel("$t$ [ps]")
ax1.set_ylabel(r"angular MSD [rad$^2$]")
ax1.set_xlim(0, min(10, t[-1]))
ax1.set_ylim(0, 10)
ax1.tick_params(direction="in")
ax1.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig("rotational_diffusion.png", bbox_inches="tight")
plt.show()
print(
    f"[dipole] frames={rot_res['n_frames_used']} (ROT_N_FRAMES={ROT_N_FRAMES!r})"
)
print(
    f"[dipole] Dr(C1)={rot_res['Dr_C1_per_ps']:.4f}, "
    f"geodesic={rot_res['Dr_amsd_per_ps']:.4f}, "
    f"unwrap={rot_res['Dr_amsd_unwrapped_per_ps']:.4f} /ps"
)


## 5. 转动扩散（完整姿态 body frame）

`orientation="body"`：由 O + 两 H 建正交三轴 $R(t)$。三种量：轴平均 $C_1$、测地线 $\langle\phi^2\rangle$（$\approx 6 D_r t$）、unwrap $\langle|\Phi|^2\rangle$。

同样受 `ROT_N_FRAMES` 控制。


In [ ]:
rot_body = rdiff.rotational_diffusion(
    u_rot,
    orientation="body",
    origin_select=SELECT_O,
    target_select=SELECT_H,
    n_target=2,
    dt_fs=DT_ROT_FS,
    n_frames=ROT_N_FRAMES,
    c1_t_min_ps=0.5, c1_t_max_ps=5.0,
    amsd_t_min_ps=0.2, amsd_t_max_ps=2.0,
)

tb = rot_body["times_ps"]
c1b, c2b = rot_body["C1"], rot_body["C2"]
amsd_b, amsd_bu = rot_body["angular_msd"], rot_body["angular_msd_unwrapped"]
slb = rot_body["fit_slice_c1"]
slb2 = rot_body["fit_slice_amsd"]
slbu = rot_body["fit_slice_amsd_unwrapped"]
pref = rot_body["amsd_prefactor"]  # 6 for body frame

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 3.2))
ax0.plot(tb, c1b, "-", label=r"axis-avg $C_1$")
ax0.plot(tb, c2b, "-", label=r"axis-avg $C_2$")
tt = tb[slb]
ax0.plot(
    tt, rot_body["C1_amp"] * np.exp(-2.0 * rot_body["Dr_C1_per_ps"] * tt),
    "k--", lw=1,
    label=rf"$D_r(C_1)={rot_body['Dr_C1_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
ax0.set_xlabel("$t$ [ps]")
ax0.set_ylabel("body-axis orientation correlation")
ax0.set_xlim(0, min(20, tb[-1]))
ax0.tick_params(direction="in")
ax0.legend(frameon=False, fontsize=8)

ax1.plot(tb, amsd_b, "-", label=r"geodesic $\langle\phi^2\rangle$")
ax1.plot(tb, amsd_bu, "-", alpha=0.75, label=r"unwrap $\langle|\Phi|^2\rangle$")
slope = pref * rot_body["Dr_amsd_per_ps"]
tt2 = tb[slb2]
ax1.plot(
    tt2, slope * tt2 + (amsd_b[slb2][0] - slope * tt2[0]),
    "k--", lw=1,
    label=rf"$D_r(\phi^2)={rot_body['Dr_amsd_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
slope_u = pref * rot_body["Dr_amsd_unwrapped_per_ps"]
ttu = tb[slbu]
ax1.plot(
    ttu, slope_u * ttu + (amsd_bu[slbu][0] - slope_u * ttu[0]),
    ":", color="tab:orange", lw=1.2,
    label=rf"$D_r(\Phi)={rot_body['Dr_amsd_unwrapped_per_ps']:.3f}\,\mathrm{{ps}}^{{-1}}$",
)
ax1.set_xlabel("$t$ [ps]")
ax1.set_ylabel(r"body angular MSD [rad$^2$]")
ax1.set_xlim(0, min(10, tb[-1]))
ax1.set_ylim(0, 10)
ax1.tick_params(direction="in")
ax1.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig("rotational_diffusion_body.png", bbox_inches="tight")
plt.show()
print(
    f"[body] frames={rot_body['n_frames_used']} (ROT_N_FRAMES={ROT_N_FRAMES!r})"
)
print(
    f"[body] Dr(C1)={rot_body['Dr_C1_per_ps']:.4f}, "
    f"geodesic={rot_body['Dr_amsd_per_ps']:.4f}, "
    f"unwrap={rot_body['Dr_amsd_unwrapped_per_ps']:.4f} /ps "
    f"(MSD pref={pref:g})"
)


## 6. 粘度（Green–Kubo）

读相关文件最后一块；$V$、$T$ 取生产段平均。


In [ ]:
if "prod" not in globals():
    prod = read_result_thermo("log.lammps", segment=-1)

corr = visc_an.read_ave_correlate_last("result_viscosity_correlate.dat")
V = float(prod["volume"].mean())
T = float(prod["temp"].mean())
visc = visc_an.integrate_gk_viscosity(
    corr, volume_A3=V, temperature_K=T,
    nevery=5, dt_fs=1.0, t_cut_fs=2000.0,
)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 3.2))
ax0.plot(visc["times_fs"] / 1000.0, visc["corr_mean"], "-")
ax0.axhline(0.0, color="k", lw=0.6)
ax0.set_xlabel("$t$ [ps]")
ax0.set_ylabel(r"$\langle P_{\alpha\beta}(0)P_{\alpha\beta}(t)\rangle$ [atm$^2$]")
ax0.tick_params(direction="in")
ax1.plot(visc["times_fs"] / 1000.0, visc["running_eta_cP"], "-")
ax1.axhline(visc["eta_cP"], color="k", ls="--", lw=0.8)
ax1.set_xlabel(r"$t_\mathrm{cut}$ [ps]")
ax1.set_ylabel(r"running $\eta$ [cP]")
ax1.tick_params(direction="in")
fig.tight_layout()
fig.savefig("viscosity_gk.png", bbox_inches="tight")
plt.show()
print(f"V={V:.1f} Å³, T={T:.2f} K, η={visc['eta_cP']:.3f} cP")


## 7. 结果汇总


In [ ]:
summary = pd.DataFrame([
    {"quantity": "T_prod", "value": prod["temp"].mean(), "unit": "K"},
    {"quantity": "P_prod", "value": prod["press"].mean(), "unit": "atm"},
    {"quantity": "density_prod", "value": prod["density"].mean(), "unit": "g/cm^3"},
    {"quantity": "D_O", "value": trans_O["D_1e5_cm2_s"], "unit": "1e-5 cm^2/s"},
    {"quantity": "D_COM", "value": trans_COM["D_1e5_cm2_s"], "unit": "1e-5 cm^2/s"},
    {"quantity": "Dr_dipole_C1", "value": rot_res["Dr_C1_per_ps"], "unit": "1/ps"},
    {"quantity": "Dr_dipole_amsd", "value": rot_res["Dr_amsd_per_ps"], "unit": "1/ps"},
    {"quantity": "Dr_dipole_unwrap", "value": rot_res["Dr_amsd_unwrapped_per_ps"], "unit": "1/ps"},
    {"quantity": "Dr_body_C1", "value": rot_body["Dr_C1_per_ps"], "unit": "1/ps"},
    {"quantity": "Dr_body_amsd", "value": rot_body["Dr_amsd_per_ps"], "unit": "1/ps"},
    {"quantity": "Dr_body_unwrap", "value": rot_body["Dr_amsd_unwrapped_per_ps"], "unit": "1/ps"},
    {"quantity": "eta_GK", "value": visc["eta_cP"], "unit": "cP"},
])
display(summary)
summary.to_csv("summary_transport.csv", index=False)
print("saved: summary_transport.csv")
print(f"TRANS_N_FRAMES = {TRANS_N_FRAMES!r}  (None = full trans trajectory)")
print(f"ROT_N_FRAMES = {ROT_N_FRAMES!r}  (None = full rot trajectory)")
